# Executive Summary

In this notebook, I will do a step-by-step exploration of the News API v2/everything endpoint, building from bare keyword lookup to multi-operator domain filter queries.

I then try to apply this to 12 representative companies extracted from the 96 company database.

I will also save any extracted results under `data/raw/search_cache` For future reference and to create a historical collection.

IMPORTANT DESIGN CHOICHES:
- Given Vishal's previous limitations in selecting and finding news for SMEs, I will try a different approach and create a **scoring heuristic** that rewards companies that are more likely to be covered by the news. I hereby acknowledge that this is going to bias the results, so I will also include a randomly selected sample with a low heuristic.

### 1. Setup

In [7]:
import importlib, subprocess, sys

def ensure(pkg, import_as=None):
    try: importlib.import_module(import_as or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

ensure('beautifulsoup4', 'bs4')
ensure('lxml')

In [8]:
import os, json, time, re
from pathlib import Path
from datetime import datetime, timedelta, date
from urllib.parse import urlencode, quote_plus, urlparse, parse_qs, unquote

import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv('../.env')

NEWS_API_KEY = os.getenv('NEWS_API_KEY')
assert NEWS_API_KEY, 'NEWS_API_KEY missing from .env'

TODAY      = date.today()
MONTH_AGO  = TODAY - timedelta(days=28)   # stay inside free-tier 30-day window
EVERYTHING_URL = 'https://newsapi.org/v2/everything'

print(f'Key loaded. Window: {MONTH_AGO} → {TODAY}')

Key loaded. Window: 2026-06-07 → 2026-07-05


In [ ]:
CACHE_DIR = Path('../data/raw/search_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

_request_count = 0

def cached_get(url: str, params: dict, company_number: str, label: str) -> dict:
    '''GET with disk cache. Never hits the API twice for the same (company_number, label).'''
    global _request_count
    cache_path = CACHE_DIR / f'{company_number}_newsapi_{label}.json'
    if cache_path.exists():
        return json.loads(cache_path.read_text())
    _request_count += 1
    resp = requests.get(url, params=params,
                        headers={'X-Api-Key': NEWS_API_KEY}, timeout=10)
    data = resp.json()
    cache_path.write_text(json.dumps(data, indent=2))
    print(f'  [call #{_request_count}] {company_number}/{label} '
          f'→ status={data.get("status")} total={data.get("totalResults","?")} ')
    return data

def show_articles(data: dict, n: int = 5):
    '''Pretty-print top-n articles from a NewsAPI response dict.'''
    if data.get('status') != 'ok':
        print(f'ERROR: {data.get("code")} - {data.get("message")}')
        return
    total = data.get('totalResults', 0)
    arts  = data.get('articles', [])
    print(f'totalResults: {total}  (showing {min(n, len(arts))})')
    print('-' * 80)
    for a in arts[:n]:
        pub  = a.get('publishedAt', '')[:10]
        src  = (a.get('source') or {}).get('name', 'unknown')
        desc = (a.get('description') or '')[:120]
        print(f'[{pub}] {src}')
        print(f'  {a.get("title", "")}')
        print(f'  {desc}')
        print()

### 2. Company Selection + Heuristic

The heuristic follows this list:

1. If company.category = "public limited company", it gets 4 points because PLCs have disclosure obligations and are almost guaranteed to be covered by the press.
2. If account.category = group, it will give 3 points because the company files consolidated group accounts = is the parent of a corporate group with subsidiaries = more coverage signal
3. If num.mort.charges > 0 the company gets 2 points because registered charge = financial hitory = more business events = more press
4. Age will be a continous normalised value 0-1 as older firm have longer press history to draw from

In [10]:
df = pd.read_csv('../data/processed/nb04_stress_test_sample.csv')
df.columns = [c.strip() for c in df.columns]

df['_plc']   = (df['CompanyCategory'] == 'Public Limited Company').astype(int) * 4
df['_group'] = (df['Accounts.AccountCategory'] == 'GROUP').astype(int) * 3
df['_mort']  = (df['Mortgages.NumMortCharges'] > 0).astype(int) * 2
df['_inc']   = pd.to_datetime(df['IncorporationDate'], dayfirst=True, errors='coerce')
df['_age']   = (pd.Timestamp(str(TODAY)) - df['_inc']).dt.days / 365.25

picks_list = []
for (sector, segment), grp in df.groupby(['sector', 'segment']):
    grp = grp.copy()
    mn, mx = grp['_age'].min(), grp['_age'].max()
    grp['_age_n'] = (grp['_age'] - mn) / (mx - mn) if mx > mn else 0.0
    grp['_score'] = grp['_plc'] + grp['_group'] + grp['_mort'] + grp['_age_n']
    picks_list.append(grp.loc[grp['_score'].idxmax()])

picks = pd.DataFrame(picks_list).reset_index(drop=True)
disp = picks[['CompanyName','CompanyNumber','RegAddress.PostTown','sector','segment',
              '_age','_plc','_group','_mort','_score']]
disp.columns = ['Name','Number','Town','Sector','Tier','Age','PLC','Grp','Mort','Score']
print(disp.to_string(index=False))

                                 Name   Number           Town                           Sector   Tier        Age  PLC  Grp  Mort     Score
                           WHALAR LTD 09803195         LONDON           Fast growth & emerging  Large  10.759754    0    3     2  5.560240
RADIO COMPUTING SERVICES (UK) LIMITED 02844235      GODALMING           Fast growth & emerging Medium  32.898015    0    0     2  3.000000
  J.C. COMPUTER SERVICES (UK) LIMITED 02254984         HARROW           Fast growth & emerging  Micro  38.151951    0    0     2  3.000000
                         VEREMARK LTD 11681510         LONDON           Fast growth & emerging  Small   7.633128    0    0     2  2.135820
                       ZENTIA LIMITED 00207732    TYNE & WEAR                    Manufacturing  Large 100.908966    0    0     2  3.000000
        SHEPCOTE DISTRIBUTORS LIMITED 00949706      DRIFFIELD                    Manufacturing Medium  57.316906    0    0     2  3.000000
                PECO SERVIC

### 3. NewsAPI Architecture

News API has three types of endpoints:

1. V2 Everything, which searches all 150,000 indexed articles.
2. V2 Top Headlines, which covers breaking news for a country or a category.
3. V2 Top Headlines Sources, which gives metadata about available publishers.

#### 3.1 Level 1 - Bare query

I start with the simplest possible call, which queries the name (`q={name}`). I try with the company that scored the highest heuristic, NCC Group.

In [11]:
# NCC Group PLC: highest-confidence pick (PLC, cybersecurity, FTSE-listed)
data_ncc_l1 = cached_get(
    url=EVERYTHING_URL,
    params={'q': 'NCC Group', 'pageSize': 10},
    company_number='04627044',
    label='L1_bare'
)
show_articles(data_ncc_l1)

totalResults: 53  (showing 5)
--------------------------------------------------------------------------------
[2026-06-11] MarketBeat
  NCC Group H1 Earnings Call Highlights
  NCC Group (LON:NCC) said it has completed a major strategic reset after selling Escode and is now operating as a focused

[2026-06-24] Infosecurity Magazine
  Iran-Linked MuddyWater Poses as Ransomware Gang to Mask Cyber Espionage
  An NCC Group report warns state-backed hackers are attempting to hide activity by posing as ransomware groups and deploy

[2026-06-11] Internet
  The Gentlemen Ransomware Claims 478 Victims, Can Spread Like a Worm
  A new analysis of The Gentlemen operation has revealed that the financially motivated threat group initially operated as

[2026-06-18] NPR
  As America turns 250, one museum makes history possible to touch
  Federal law requires most museums and other buildings to be accessible to people with disabilities. But access to what's

[2026-06-25] Internet
  ThreatsDay Bulletin:

In [ ]:
# PECO SERVICES LIMITED - manufacturing micro-firm, Kirkby Stephen.
# 'PECO' is also PECO Energy, a large Philadelphia utility with heavy US press coverage.
# This is the canonical false-positive example throughout this notebook.
data_peco_l1 = cached_get(
    url=EVERYTHING_URL,
    params={'q': 'Peco', 'pageSize': 10},
    company_number='04623078',
    label='L1_bare'
)
show_articles(data_peco_l1)
# Expected: mostly PECO Energy (Philadelphia utility) articles.

totalResults: 42  (showing 5)
--------------------------------------------------------------------------------
[2026-05-30] Xatakamovil.com
  He creado una app para mi Android en quince minutos sin tener ni idea de programación. Ahora puedo hacer cualquier idea que me apetezca
  Cuántas veces habré buceado por Play Store buscando una app muy concreta solo para terminar descargando opciones llenas 

[2026-06-05] Prtimes.jp
  「子どもの時間感覚の育て方」「学び旅＆体験ガイド」2大特集／『AERA with Kids2026夏号』発売！／特別付録「変身！正方形パズル」／教科別おすすめドリル／巻頭インタビューpecoさん登場
  株式会社朝日新聞出版のプレスリリース（2026年6月5日 11時00分）「子どもの時間感覚の育て方」「学び旅＆体験ガイド」2大特集／『AERA with Kids2026夏号』発売！／特別付録「変身！正方形パズル」／教科別おすすめドリル／巻

[2026-06-02] PRNewswire
  Façonnons l'avenir ensemble ! Voilà ce qu'on dit les participants du monde entier lors de la réunion 2026 des dirigeants locaux Chine-PECO et de la semaine des villes amies de Shandong
  JINAN, Chine, 2 juin 2026 /PRNewswire/ -- Du 25 au 29 mai 2026, la 7e réunion des dirigeants locaux Chine-PECO et la sem

[2026-06-02] 

#### 3.2 Level 2 - language filter + date window

To increase the filter accuracy, one can select a specific language and a specific date window. These are two additions that cost nothing but reduce noise meaningfully.

I will also experiment with the sort by options, which can be by:
1. Relevancy = closer term match first using TF-IDF (Term Frequency-Inverse Document Frequency)
2. Popularity = higher traffic sources
3. Published at = newest first

I will mainly use **Relevancy**

In [13]:
data_ncc_l2 = cached_get(
    url=EVERYTHING_URL,
    params={'q': 'NCC Group', 'language': 'en',
            'from': str(MONTH_AGO), 'to': str(TODAY),
            'sortBy': 'relevancy', 'pageSize': 10},
    company_number='04627044',
    label='L2_lang_date'
)
show_articles(data_ncc_l2)

totalResults: 22  (showing 5)
--------------------------------------------------------------------------------
[2026-06-11] MarketBeat
  NCC Group H1 Earnings Call Highlights
  NCC Group (LON:NCC) said it has completed a major strategic reset after selling Escode and is now operating as a focused

[2026-06-24] Infosecurity Magazine
  Iran-Linked MuddyWater Poses as Ransomware Gang to Mask Cyber Espionage
  An NCC Group report warns state-backed hackers are attempting to hide activity by posing as ransomware groups and deploy

[2026-06-11] Internet
  The Gentlemen Ransomware Claims 478 Victims, Can Spread Like a Worm
  A new analysis of The Gentlemen operation has revealed that the financially motivated threat group initially operated as

[2026-06-18] NPR
  As America turns 250, one museum makes history possible to touch
  Federal law requires most museums and other buildings to be accessible to people with disabilities. But access to what's

[2026-06-25] Internet
  ThreatsDay Bulletin:

In [14]:
# sortBy=publishedAt vs relevancy: for the same query, compare top 5.
data_ncc_pub = cached_get(
    url=EVERYTHING_URL,
    params={'q': 'NCC Group', 'language': 'en',
            'from': str(MONTH_AGO), 'sortBy': 'publishedAt', 'pageSize': 5},
    company_number='04627044',
    label='L2_publishedAt'
)
print('--- sortBy=publishedAt ---')
show_articles(data_ncc_pub)

--- sortBy=publishedAt ---
totalResults: 22  (showing 5)
--------------------------------------------------------------------------------
[2026-06-25] Internet
  ThreatsDay Bulletin: Smart TV Proxyware, 24-Year curl Bug, AI Crime Forums + 13 More Stories
  It’s dumb out there again.

This week has the usual smell of prod on fire and nobody wanting to admit who left the door 

[2026-06-24] Infosecurity Magazine
  Iran-Linked MuddyWater Poses as Ransomware Gang to Mask Cyber Espionage
  An NCC Group report warns state-backed hackers are attempting to hide activity by posing as ransomware groups and deploy

[2026-06-20] Seclists.org
  NSE submission: detect & enumerate AI infrastructure (MCP servers + LLM inference APIs)
  Posted by Ben Williams via dev on Jun 20Hi,
I've opened a PR adding three NSE scripts and two shared nselibs to detect a

[2026-06-18] NPR
  As America turns 250, one museum makes history possible to touch
  Federal law requires most museums and other buildings to be ac

In [17]:
# sortBy=publishedAt vs relevancy: for the same query, compare top 5.
data_peco_pub = cached_get(
    url=EVERYTHING_URL,
    params={'q': 'Peco', 'language': 'en',
            'from': str(MONTH_AGO), 'sortBy': 'publishedAt', 'pageSize': 5},
    company_number='04623078',
    label='L2_publishedAt'
)
print('--- sortBy=publishedAt ---')
show_articles(data_peco_pub)

--- sortBy=publishedAt ---
totalResults: 8  (showing 5)
--------------------------------------------------------------------------------
[2026-07-04] Biztoc.com
  Union representing about 1,600 PECO workers go on strike
  The union representing about 1,600 PECO workers went on strike at midnight Saturday after contract negotiations failed t

[2026-06-25] Biztoc.com
  PECO workers to go on strike July 4, union officials say
  The union representing PECO workers announced Thursday that they will go on strike on July 4.
IBEW Local 614 -- which re

[2026-06-25] GlobeNewswire
  PECO Pallet Selected by Canadian Institute of Traffic and Transportation (CITT) for 2026 Environmental Sustainability Award
  PECO Pallet was recognized for the second consecutive year by the Canadian Institute of Traffic and Transportation for i

[2026-06-18] GlobeNewswire
  PECO Pallet Named to Inbound Logistics 2026 “Green 75”
  PECO Pallet for the sixth consecutive year has been named an Inbound Logistics Green 7

#### What Level 2 improved over Level 1

Level 1 was a bare `q={name}` call - no filters, default sort. Level 2 adds three near-zero-cost parameters (`language`, `from`/`to`, `sortBy`) that cut noise without touching the query itself.

**What changed:**

| Lever | L1 | L2 | Effect |
|---|---|---|---|
| `language` | none | `en` | Drops non-English hits. NCC's raw 53 fell to 22 - the removed articles were Japanese/French/Romanian PR wire copy (visible in the Peco L1 output). |
| `from` / `to` | none (default ~1 month, undated) | explicit 28-day window | Pins results to a known, reproducible window that stays inside the free-tier 30-day limit; makes counts comparable across companies and reruns. |
| `sortBy` | default (`publishedAt`) | `relevancy` | Surfaces closest term matches first (TF-IDF) rather than just newest, so the on-target NCC earnings/report articles float to the top instead of unrelated same-week stories. |

**Result:** NCC Group from 53 to 22 results, with the noise (e.g. the NPR museum piece, foreign-language wire) pushed down or removed. The `sortBy=publishedAt` vs `relevancy` comparison confirms the difference: `publishedAt` leads with a generic "ThreatsDay Bulletin" and a Nigerian airtime story, while `relevancy` leads with the actual NCC Group earnings call and threat report.

**What Level 2 does *not* fix:** the core disambiguation problem. Peco (the Kirkby Stephen micro-firm) still returns PECO Energy / China-PECO summit articles - those are English, recent, and relevant *to the wrong entity*. Filtering by language and date cannot separate a name collision; that requires query-level operators and verification.

#### 3.3 Level 3 - exact phrase matching

For this level, I experiment with the difference between wrapping a phrase in double quotes (which forces an exact substring match) or having it unquoted.

In [19]:
# NCC Group: unquoted vs quoted
data_ncc_unq = cached_get(
    EVERYTHING_URL,
    {'q': 'NCC Group', 'language': 'en', 'from': str(MONTH_AGO),
     'sortBy': 'relevancy', 'pageSize': 5},
    '04627044', 'L3_unquoted'
)
data_ncc_q = cached_get(
    EVERYTHING_URL,
    {'q': '"NCC Group"', 'language': 'en', 'from': str(MONTH_AGO),
     'sortBy': 'relevancy', 'pageSize': 5},
    '04627044', 'L3_quoted'
)
print(f'Unquoted: {data_ncc_unq.get("totalResults",0)}')
print(f'Quoted: {data_ncc_q.get("totalResults",0)}')
print()
show_articles(data_ncc_q)

Unquoted: 22
Quoted: 9

totalResults: 9  (showing 5)
--------------------------------------------------------------------------------
[2026-06-11] MarketBeat
  NCC Group H1 Earnings Call Highlights
  NCC Group (LON:NCC) said it has completed a major strategic reset after selling Escode and is now operating as a focused

[2026-06-24] Infosecurity Magazine
  Iran-Linked MuddyWater Poses as Ransomware Gang to Mask Cyber Espionage
  An NCC Group report warns state-backed hackers are attempting to hide activity by posing as ransomware groups and deploy

[2026-06-11] Internet
  The Gentlemen Ransomware Claims 478 Victims, Can Spread Like a Worm
  A new analysis of The Gentlemen operation has revealed that the financially motivated threat group initially operated as

[2026-06-25] Internet
  ThreatsDay Bulletin: Smart TV Proxyware, 24-Year curl Bug, AI Crime Forums + 13 More Stories
  It’s dumb out there again.

This week has the usual smell of prod on fire and nobody wanting to admit who left

In [21]:
# PECO false-positive fix.
# Quoted exact phrase eliminates the PECO Energy articles.
# Zero results = correct outcome for an obscure Cumbrian firm.
data_peco_q = cached_get(
    EVERYTHING_URL,
    {'q': '"Peco Services"', 'language': 'en', 'from': str(MONTH_AGO), 'pageSize': 5},
    '04623078', 'L3_quoted'
)
print(f'Bare "Peco": {data_peco_l1.get("totalResults",0)} results (mostly PECO Energy)')
print(f'Quoted "Peco Services": {data_peco_q.get("totalResults",0)} results')
show_articles(data_peco_q)

Bare "Peco": 42 results (mostly PECO Energy)
Quoted "Peco Services": 0 results
totalResults: 0  (showing 0)
--------------------------------------------------------------------------------


#### 3.4 Boolean operatores

In the new API query request, you can add a series of Boolean operators to the parameter q. These mainly include:

1. AND: Means both terms are required
2. NOT: Means the term must not appear
3. OR: Means at least one
4. (A or B):  this allows for standard Boolean grouping.

In [22]:
# AND: sector keyword filters to articles *about* NCC in cybersecurity context
data_ncc_bool = cached_get(
    EVERYTHING_URL,
    {'q': '"NCC Group" AND (cybersecurity OR "cyber security" OR breach OR vulnerability)',
     'language': 'en', 'from': str(MONTH_AGO), 'sortBy': 'relevancy', 'pageSize': 10},
    '04627044', 'L4_bool_sector'
)
show_articles(data_ncc_bool)

totalResults: 6  (showing 5)
--------------------------------------------------------------------------------
[2026-06-11] MarketBeat
  NCC Group H1 Earnings Call Highlights
  NCC Group (LON:NCC) said it has completed a major strategic reset after selling Escode and is now operating as a focused

[2026-06-24] Infosecurity Magazine
  Iran-Linked MuddyWater Poses as Ransomware Gang to Mask Cyber Espionage
  An NCC Group report warns state-backed hackers are attempting to hide activity by posing as ransomware groups and deploy

[2026-06-11] Internet
  The Gentlemen Ransomware Claims 478 Victims, Can Spread Like a Worm
  A new analysis of The Gentlemen operation has revealed that the financially motivated threat group initially operated as

[2026-06-25] Internet
  ThreatsDay Bulletin: Smart TV Proxyware, 24-Year curl Bug, AI Crime Forums + 13 More Stories
  It’s dumb out there again.

This week has the usual smell of prod on fire and nobody wanting to admit who left the door 

[2026-06-11]

In [23]:
# OR: ZENTIA LIMITED renamed from Armstrong World Industries in 2021.
# Press articles from before the rename still use the old name.
# OR bridges the gap without losing either trail.
data_zentia = cached_get(
    EVERYTHING_URL,
    {'q': '"Zentia" OR "Armstrong World Industries"',
     'language': 'en', 'from': str(MONTH_AGO), 'sortBy': 'relevancy', 'pageSize': 10},
    '00207732', 'L4_or_prev_name'
)
show_articles(data_zentia)

totalResults: 6  (showing 5)
--------------------------------------------------------------------------------
[2026-06-12] Yahoo Entertainment
  Armstrong World Industries’ (AWI) Growth Initiatives Regaining Traction
  

[2026-06-11] PRNewswire
  INVESTOR ALERT: Pomerantz Law Firm Investigates Claims On Behalf of Investors of Armstrong World Industries, Inc. - AWI
  NEW YORK, June 11, 2026 /PRNewswire/ -- Pomerantz LLP is investigating claims on behalf of investors of Armstrong World 

[2026-06-08] PRNewswire
  AWI Investors Have Opportunity to Join Armstrong World Industries, Inc. Fraud Investigation with the Schall Law Firm
  LOS ANGELES, June 8, 2026 /PRNewswire/ -- The Schall Law Firm, a national shareholder rights litigation firm, announces 

[2026-06-12] Biztoc.com
  Armstrong World Industries’ (AWI) Growth Initiatives Regaining Traction
  In its first-quarter 2026 investor letter, The London Company Small-Mid Cap Strategy highlighted Armstrong World Industr

[2026-06-09] GlobeNe

In [24]:
# NOT: Nielsen Book Services — 'Nielsen' alone matches Nielsen Holdings, Nielsen ratings,
# Nielsen-Norman Group, etc. Quoted 'Nielsen Book' + NOT eliminates media/ratings FPs.
data_nielsen = cached_get(
    EVERYTHING_URL,
    {'q': '"Nielsen Book" NOT (ratings OR television OR audience OR panel)',
     'language': 'en', 'from': str(MONTH_AGO), 'sortBy': 'relevancy', 'pageSize': 10},
    '00070437', 'L4_not'
)
show_articles(data_nielsen)

totalResults: 1  (showing 1)
--------------------------------------------------------------------------------
[2026-06-26] Publishingperspectives.com
  China Bestsellers, May 2026: Screen Adaptations Drive Book Sales
  OpenBook shares insight into May 2026 book sales across fiction, nonfiction, and children’s titles, with insight into wh



#### 3.5 level 5 - the `searchIn` parameter

By default, NewsAPI matches the query against title + description (i.e. lead paragraph opf the article) + content simultaneously. 

However, using the searchIn parameter, you can specify:
1. **title**: if you want the company name to appear just in the **headline** for maximum precision and a lower recall.
2. **title,descritpion**: for a balanced approach broadening the search to prominent mentions in the lead paragraph.

In [25]:
base = {'language': 'en', 'from': str(MONTH_AGO), 'sortBy': 'relevancy', 'pageSize': 5}

for si, lbl in [('title', 'L5_title'), ('title,description', 'L5_titledesc'), (None, 'L5_all')]:
    p = {**base, 'q': '"NCC Group"'}
    if si: p['searchIn'] = si
    d = cached_get(EVERYTHING_URL, p, '04627044', lbl)
    label_str = si or 'all fields'
    print(f'searchIn={label_str:<22}: totalResults: {d.get("totalResults",0)}')

searchIn=title                 : totalResults: 2
searchIn=title,description     : totalResults: 4
searchIn=all fields            : totalResults: 9


In [26]:
# Whalar: distinctive name but may appear as a passing mention in broad influencer-marketing
# articles. searchIn=title isolates articles written specifically about Whalar.
data_w_title = cached_get(EVERYTHING_URL,
    {'q': '"Whalar"', 'language': 'en', 'from': str(MONTH_AGO), 'searchIn': 'title', 'pageSize': 5},
    '09803195', 'L5_title')
data_w_all = cached_get(EVERYTHING_URL,
    {'q': '"Whalar"', 'language': 'en', 'from': str(MONTH_AGO), 'pageSize': 5},
    '09803195', 'L5_all')
print(f'Whalar searchIn=title: {data_w_title.get("totalResults",0)}')
print(f'Whalar searchIn=all: {data_w_all.get("totalResults",0)}')
show_articles(data_w_title)

Whalar searchIn=title: 11
Whalar searchIn=all: 28
totalResults: 11  (showing 5)
--------------------------------------------------------------------------------
[2026-06-10] Business Insider
  Why Accenture buying Whalar is a 'coming-of-age moment' for creator marketing
  Accenture Song's acquisition of the creator and social agency Whalar is more evidence that big budgets are shifting to i

[2026-06-12] Digiday
  Future of Marketing Briefing: Accenture’s Whalar bet: own the room when creator marketing gets complicated
  The Whalar deal is Accenture running the same play it ran on programmatic — only this time it got there earlier.

[2026-06-13] Yahoo Entertainment
  Accenture (ACN) to Acquire Whalar, Expanding Accenture Song into Creator Economy
  Accenture (NYSE:ACN) is one of the most undervalued quality stocks to invest in. On June 8, Accenture announced an agree

[2026-06-08] Adweek
  EXCLUSIVE: Accenture Song Will Buy Whalar, Gaining Global Scale in Influencer Marketing
  Accentu

#### 3.6 level 6 - domains and excludeDmains

News API indexes 5,000+ sources. Some of these are irrelevant to the UK commercial environment, and by filtering by language, we are already sorting out many of these.

However, we could restrict the search to specific UK business press. This would definitely help for large/medium firms, but for micro/small firms, it could kill the local or trade press coverage.

Probably the best approach is to use this very feature only on specific firm sizes.

In [27]:
# UK business and tech press — suitable for Large/Medium firms
UK_LARGE_DOMAINS = ','.join([
    'ft.com', 'theguardian.com', 'bbc.co.uk', 'reuters.com', 'cityam.com',
    'businesscloud.co.uk', 'themanufacturer.com', 'businesslive.co.uk',
    'growthbusiness.co.uk', 'uktech.news', 'techcrunch.com',
    'computerweekly.com', 'theregister.com', 'silicon.co.uk',
])
# UK regional press — extend for Medium firms in specific geographies
UK_REGIONAL_DOMAINS = ','.join([
    'manchestereveningnews.co.uk', 'yorkshirepost.co.uk',
    'birminghampost.co.uk', 'scotsman.com', 'walesonline.co.uk',
    'chroniclelive.co.uk', 'thestar.co.uk',
])

data_ncc_dom = cached_get(
    EVERYTHING_URL,
    {'q': '"NCC Group"', 'language': 'en', 'from': str(MONTH_AGO),
     'sortBy': 'relevancy', 'searchIn': 'title,description',
     'domains': UK_LARGE_DOMAINS, 'pageSize': 10},
    '04627044', 'L6_domains'
)
print('UK business domain filter:')
show_articles(data_ncc_dom)

UK business domain filter:
totalResults: 2  (showing 2)
--------------------------------------------------------------------------------
[2026-06-11] ComputerWeekly.com
  NCC Group outlines cyber future
  Firm concludes strategic review, ruling out a sale, and will operate as a security and services player

[2026-06-12] ComputerWeekly.com
  Channel catch-up: News in brief
  Developments this week at Leaseweb, Goldilock Secure, NCC Group, ConnectWise, Smarttech247, Pax8, Ten10 Solutions and Sc



In [29]:
# excludeDomains: block specific aggregators/syndication noise without a whitelist
data_ncc_excl = cached_get(
    EVERYTHING_URL,
    {'q': '"NCC Group"', 'language': 'en', 'from': str(MONTH_AGO),
     'sortBy': 'relevancy', 'excludeDomains': 'yahoo.com,msn.com', 'pageSize': 5},
    '04627044', 'L6_excl'
)
print(f'Excluding yahoo/msn: totalResults: {data_ncc_excl.get("totalResults",0)}')

Excluding yahoo/msn: totalResults: 9


### 4. Signal templates

Using the boolean operators we can also create a dictionary of signal templates where you have a specific keyword and a pre-populated set of boolean operators. This way one can run specific query with a specific purpose. For instance, if one wants to understand growth potential of a firm, one could attach keywords such as funding, investment, expansion. 

The query would look as:

query = company name + signal templates(signal key).

In [30]:
SIGNAL_TEMPLATES = {
    'financial_distress':  '(insolvency OR administration OR "winding up" OR "county court" OR CCJ)',
    'supply_chain':        '("supply chain" OR disruption OR shortage OR logistics OR delay)',
    'growth':              '(funding OR investment OR expansion OR acquisition OR "series A" OR "series B")',
    'leadership_change':   '(appointed OR "chief executive" OR CEO OR resigned OR director)',
    'regulatory_action':   '(fine OR penalty OR investigation OR FCA OR HSE OR "trading standards")',
    'cyber_incident':      '(breach OR ransomware OR hack OR "data leak" OR vulnerability)',
    'M_and_A':             '(acquired OR merger OR takeover OR "private equity" OR buyout)',
    'credit_risk':         '(debt OR "credit rating" OR "payment default" OR insolvency)',
}

# Example: risk strategist flags supply chain disruption for NCC Group
compound_q = f'"NCC Group" AND {SIGNAL_TEMPLATES["supply_chain"]}'
print(f'Compound query: {compound_q}')

# Demo — run it
data_ncc_signal = cached_get(
    EVERYTHING_URL,
    {'q': compound_q, 'language': 'en', 'from': str(MONTH_AGO), 'pageSize': 5},
    '04627044', 'L6_signal_supply'
)
show_articles(data_ncc_signal)

Compound query: "NCC Group" AND ("supply chain" OR disruption OR shortage OR logistics OR delay)
totalResults: 3  (showing 3)
--------------------------------------------------------------------------------
[2026-06-24] Infosecurity Magazine
  Iran-Linked MuddyWater Poses as Ransomware Gang to Mask Cyber Espionage
  An NCC Group report warns state-backed hackers are attempting to hide activity by posing as ransomware groups and deploy

[2026-06-25] Internet
  ThreatsDay Bulletin: Smart TV Proxyware, 24-Year curl Bug, AI Crime Forums + 13 More Stories
  It’s dumb out there again.

This week has the usual smell of prod on fire and nobody wanting to admit who left the door 

[2026-06-12] ComputerWeekly.com
  Channel catch-up: News in brief
  Developments this week at Leaseweb, Goldilock Secure, NCC Group, ConnectWise, Smarttech247, Pax8, Ten10 Solutions and Sc

